# Entrenar y comparar todos los modelos (TP II)

Un solo notebook que corre, con el **mismo dataset, split y métrica**, los modelos que pide el enunciado y los compara:

| | modelo | dónde corre | script |
|---|---|---|---|
| a-ii | Random Forest **sin** embeddings (atributos a la antigua) | CPU | `src/busqueda_rf.py --modelo sin_emb` |
| a-i | Random Forest **con** embeddings (promedio de los tokens BERT de la palabra) | CPU | `src/busqueda_rf.py --modelo con_emb` |
| b | LSTM | **GPU** | `src/busqueda_lstm.py --modelo lstm` |
| c | LSTM bidireccional | **GPU** | `src/busqueda_lstm.py --modelo bilstm` |

**Protocolo común** (`src/protocolo.py`):
- **Split por documento**: train / val / test se parten por película, conferencia o nota de Wikinoticias (en Paenza y el sintético, que son un solo documento, por oración/`bloque`). Ninguna película ni oración queda a los dos lados. Es **estratificado**, para que las clases raras (capitalización 2 y 3, `¿`, `?`) no queden concentradas en un solo conjunto.
- **Métrica**: F1 macro por tarea a nivel palabra (inicial 2 clases, final 4, capitalización 4), más F1 por clase y matrices de confusión.
- **Selección** con validación; el **test se mide una sola vez** al final de cada modelo.

**Cada modelo hace su búsqueda de hiperparámetros** y al final guarda `preds_test.csv`; la última sección los compara (F1 macro, F1 por clase, matrices de confusión, por fuente y por largo de instancia).

**Reanudable**: si un modelo ya tiene su `preds_test.csv`, se saltea (`REHACER = True` para repetirlo). Las búsquedas ya hechas tampoco se repiten.

**Dónde correr cada cosa.** Las redes necesitan GPU: Colab (T4). Los Random Forest no usan GPU y en Colab hay pocos núcleos, así que conviene correrlos en tu PC (`CORRER_RF = True` en local) y subir las carpetas `rf_sin_emb/` y `rf_con_emb/` (con su `preds_test.csv`) a la carpeta de Drive. En Colab, `CORRER_RF` arranca en `False`.

**Archivos en la carpeta de Drive (`TP_AA`)**: `busqueda_lstm.py`, `busqueda_rf.py`, `atributos_rf.py`, `protocolo.py`, `evaluacion.py`, `rutas.py` (el de Colab, ver `colab/rutas.py`), `embeddings_palabras.npy` y `etiquetas_palabras.csv` (del mismo run de `generar_embeddings.ipynb`).

## 0. Configuración

In [ ]:
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

# --- qué correr -------------------------------------------------------------------------------------------------
RAPIDO = False          # True: prueba de humo (pocos datos, 2 épocas). Usa carpetas *_rapido y no se evalúa.
REHACER = False         # True: repetir aunque ya exista preds_test.csv
CORRER_RF = not EN_COLAB    # los RF usan CPU: en tu PC sí, en Colab no (subir sus resultados a Drive)
CORRER_REDES = True     # LSTM y BiLSTM (GPU)
# ------------------------------------------------------------------------------------------------------------------

ARCHIVOS = ("busqueda_lstm.py", "busqueda_rf.py", "atributos_rf.py", "protocolo.py", "evaluacion.py", "rutas.py",
            "embeddings_palabras.npy", "etiquetas_palabras.csv")

if EN_COLAB:
    from google.colab import drive
    try:
        drive.mount("/content/drive")
    except Exception as e:
        print("drive.mount() no funcionó:", type(e).__name__)
    CARPETA = next((p for p in (Path("/content/drive/MyDrive/TP_AA"), Path("/content/gdrive/MyDrive/TP_AA"), Path("/content/TP_AA"))
                    if p.exists()), None)
    assert CARPETA is not None, "No encuentro la carpeta TP_AA en Drive (si es compartida, agregale un acceso directo a Mi unidad)"
    CODIGO = SAL = CARPETA          # en Colab todo vive en la misma carpeta
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    CODIGO, SAL = RAIZ / "src", RAIZ / "salidas"

faltan = [a for a in ARCHIVOS if not ((CODIGO if a.endswith(".py") else SAL) / a).exists()]
assert not faltan, f"Faltan archivos: {faltan}"
sys.path.insert(0, str(CODIGO))
print("código:", CODIGO, "\nsalidas:", SAL, "\nRAPIDO:", RAPIDO, "| CORRER_RF:", CORRER_RF, "| CORRER_REDES:", CORRER_REDES)

try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no hay (las redes van a tardar mucho en CPU)")
except ImportError:
    print("torch no está instalado (solo hace falta para las redes)")

In [ ]:
import time


def correr(args, carpeta):
    """Corre un script y lo saltea si el modelo ya tiene sus predicciones de test."""
    carpeta = carpeta + ("_rapido" if RAPIDO else "")
    destino = SAL / carpeta / "preds_test.csv"
    if destino.exists() and not REHACER:
        print(f"{carpeta}: ya está hecho ({destino}). REHACER = True para repetirlo.")
        return
    cmd = f'python "{CODIGO / args.split()[0]}" ' + " ".join(args.split()[1:]) + (" --rapido" if RAPIDO else "")
    print(">>", cmd)
    t0 = time.time()
    get_ipython().system(cmd)          # se ve la salida a medida que corre
    print(f"{carpeta}: {(time.time() - t0) / 60:.1f} min")

## 1. El split (train / val / test)

Se muestra cuántas palabras de cada clase quedan en cada conjunto. Tiene que verse parecido entre columnas (si una clase queda con muy pocas palabras en val o test, aparece un `AVISO`). El mismo split lo usan los 4 modelos.

In [ ]:
import protocolo as pr

et = pr.cargar_etiquetas(SAL / "etiquetas_palabras.csv")
idx_train, idx_val, idx_test = pr.dividir(et)
print(pr.reporte_split(et, idx_train, idx_val, idx_test))
print("\nhuella del split:", pr.huella_split(idx_train, idx_val, idx_test))

## 2. Random Forest sin embeddings (CPU)

Atributos a la antigua (largo, vocales, acentos, posición, listas de interrogativos y conectores, y lo mismo de las 2 palabras vecinas a cada lado, más un one-hot de las 150 palabras más frecuentes del train). Búsqueda de `class_weight`, `max_depth` y `min_samples_leaf` por tarea, y factores por clase ajustados con validación.

In [ ]:
if CORRER_RF:
    correr("busqueda_rf.py --modelo sin_emb", "rf_sin_emb")
else:
    print("CORRER_RF = False: se usan los resultados que ya haya en", SAL / "rf_sin_emb")

## 3. Random Forest con embeddings (CPU)

La entrada es el embedding de la palabra (768 dimensiones), sin contexto. Misma búsqueda y mismo protocolo. Es el más lento de los dos RF (varios minutos por tarea).

In [ ]:
if CORRER_RF:
    correr("busqueda_rf.py --modelo con_emb", "rf_con_emb")
else:
    print("CORRER_RF = False: se usan los resultados que ya haya en", SAL / "rf_con_emb")

## 4. LSTM (GPU)

Búsqueda por etapas: **A** capas y tamaño oculto, **B** dropout / learning rate / weight decay, **C** peso de las clases y de cada tarea en la pérdida, **D** umbral por clase; después se reentrena con 3 semillas y se mide en test. Cada etapa parte de la mejor de la anterior y solo cambia un valor si mejora la validación en más de 0,005.

In [ ]:
if CORRER_REDES:
    correr("busqueda_lstm.py --modelo lstm --alphas", "lstm")

## 5. LSTM bidireccional (GPU)

Mismo procedimiento y mismo script; solo cambia la arquitectura (`H` es por dirección).

In [ ]:
if CORRER_REDES:
    correr("busqueda_lstm.py --modelo bilstm --alphas", "bilstm")

## 6. Comparación final en test

Se comparan solo las palabras de test que **todos** los modelos predijeron. Se calcula:
- **F1 macro** por tarea (la métrica del enunciado) y su promedio;
- **F1, precisión y recall por clase** (¿qué clase le costó más a qué modelo?);
- **matrices de confusión**, normalizadas por fila (cada fila suma 1: qué porcentaje de cada clase real se predijo como cada clase);
- **F1 por fuente** y **por largo de instancia**, para ver dónde falla cada modelo.

Tablas y figuras quedan en `salidas/evaluacion/` (para el póster).

In [ ]:
if RAPIDO:
    print("RAPIDO = True: la evaluación se hace sobre las corridas reales; poné RAPIDO = False.")
else:
    import evaluacion
    res = evaluacion.evaluar(salidas=SAL)

In [ ]:
from IPython.display import Image, display

if not RAPIDO:
    for nombre in ["comparacion_f1_macro", "confusion_punt_inicial", "confusion_punt_final", "confusion_capitalizacion",
                   "f1_por_clase_punt_inicial", "f1_por_clase_punt_final", "f1_por_clase_capitalizacion"]:
        display(Image(filename=str(SAL / "evaluacion" / f"{nombre}.png")))

### Cómo leer los resultados

- **Matriz de confusión**: la diagonal es el recall de cada clase. Un modelo que "siempre dice nada" tiene un 1,00 en la primera columna y 0 en las demás: mucha exactitud y F1 macro bajo.
- **F1 macro** pesa igual a todas las clases, por eso la capitalización 2 y 3 y el `?` (muy pocas palabras) mueven mucho el resultado y lo hacen ruidoso: mirar el F1 por clase y el número de palabras (`f1_por_clase.csv`, columna `palabras`).
- Si las clases raras del test tienen pocas palabras, diferencias de ~0,01 a 0,02 no son concluyentes.